In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

In [ ]:
# --- WaveDrom / VCD rendering utilities (auto-generated — do not edit) ---
import json, re, subprocess
from IPython.display import HTML, display

def _vcd_to_wavedrom(vcd_path, max_cycles=80, signals=None):
    """
    Minimal VCD → WaveDrom JSON converter.
    Works for single-bit and multi-bit signals from iverilog output.
    """
    with open(vcd_path) as f:
        vcd = f.read()

    # Parse variable declarations
    var_map = {}  # id_code → (name, width)
    for m in re.finditer(
        r"\$var\s+\w+\s+(\d+)\s+(\S+)\s+(\S+)", vcd
    ):
        width, code, name = int(m.group(1)), m.group(2), m.group(3)
        if signals is None or name in signals:
            var_map[code] = (name, width)

    if not var_map:
        print(f"⚠  No matching signals in {vcd_path}")
        return None

    # Parse value changes
    changes = {}  # id_code → [(time, value), ...]
    current_time = 0
    for line in vcd.splitlines():
        line = line.strip()
        if line.startswith("#"):
            current_time = int(line[1:])
        elif line.startswith("b"):
            parts = line.split()
            if len(parts) == 2 and parts[1] in var_map:
                val = int(parts[0][1:], 2) if parts[0][1:] else 0
                changes.setdefault(parts[1], []).append((current_time, val))
        elif len(line) >= 2 and line[0] in "01xXzZ" and line[1:] in var_map:
            code = line[1:]
            val = {"0": 0, "1": 1, "x": "x", "X": "x", "z": "z", "Z": "z"}[line[0]]
            changes.setdefault(code, []).append((current_time, val))

    # Determine time step (smallest non-zero delta)
    all_times = sorted({t for ch in changes.values() for t, _ in ch})
    if len(all_times) < 2:
        return None
    deltas = [all_times[i+1] - all_times[i] for i in range(len(all_times)-1) if all_times[i+1] != all_times[i]]
    if not deltas:
        return None
    step = min(deltas)
    end_time = min(all_times[-1], step * max_cycles) if max_cycles else all_times[-1]
    sample_times = list(range(0, end_time + 1, step))[:max_cycles]

    # Build WaveDrom signal list
    wave_signals = []
    for code, (name, width) in sorted(var_map.items(), key=lambda x: x[1][0]):
        ch = sorted(changes.get(code, []))
        # Sample at each time point
        samples = []
        cur_val = 0
        ci = 0
        for t in sample_times:
            while ci < len(ch) and ch[ci][0] <= t:
                cur_val = ch[ci][1]
                ci += 1
            samples.append(cur_val)

        if width == 1:
            # Single-bit: WaveDrom wave string
            wave_str = ""
            for v in samples:
                if v == "x":
                    wave_str += "x"
                elif v == "z":
                    wave_str += "z"
                else:
                    wave_str += str(int(v))
            wave_signals.append({"name": name, "wave": wave_str})
        else:
            # Multi-bit: use '=' for data with labels
            wave_str = ""
            data = []
            prev = None
            for v in samples:
                if v == prev:
                    wave_str += "."
                else:
                    wave_str += "="
                    data.append(f"0x{v:X}" if isinstance(v, int) else str(v))
                    prev = v
            wave_signals.append({"name": name, "wave": wave_str, "data": data})

    return {"signal": wave_signals, "config": {"hscale": 1}}


def show_waves(vcd_path="dump.vcd", max_cycles=80, signals=None, width=900):
    """Render VCD waveforms inline using WaveDrom."""
    wd = _vcd_to_wavedrom(vcd_path, max_cycles=max_cycles, signals=signals)
    if wd is None:
        print("No waveform data to display.")
        return
    wd_json = json.dumps(wd)
    html = f"""
    <div id="wd_{id(wd):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wd):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wd):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))


def show_wavedrom(wave_dict, width=900):
    """Render a raw WaveDrom JSON dict inline."""
    wd_json = json.dumps(wave_dict)
    html = f"""
    <div id="wd_{id(wave_dict):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wave_dict):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wave_dict):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))

print("✓ WaveDrom helpers loaded — use show_waves('dump.vcd') after simulation")

# Day 11 Lab: UART Transmitter

## Overview
Build and verify a complete UART TX module, then send "HELLO" from the
Go Board to your PC terminal.

## Prerequisites
- Pre-class video on UART protocol and TX architecture
- Working debounce module from Day 5
- Working hex_to_7seg from Day 2

## Exercises

| # | Exercise | Time | Key SLOs |
|---|----------|------|----------|
| 1 | Baud Rate Generator | 15 min | 11.2 |
| 2 | UART TX Module + Testbench | 40 min | 11.3, 11.4 |
| 3 | Hardware Verification | 25 min | 11.5 |
| 4 | Multi-Character Sender | 20 min | 11.3, 11.5 |
| 5 | Parity Support (Stretch) | 15 min | 11.3 |

## Deliverables
1. Baud generator verified in simulation
2. UART TX with self-checking protocol-aware testbench (all tests pass)
3. "HELLO" received on PC terminal emulator

## Terminal Setup
| Platform | Command |
|----------|---------|
| Linux | `screen /dev/ttyUSB0 115200` |
| macOS | `screen /dev/cu.usbserial-* 115200` |
| Windows | PuTTY → COMx, 115200, 8N1 |

---
## Exercise Files

The starter files for each exercise are below. Edit the code, then run the simulation/build cells.

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile baud_gen.v
// =============================================================================
// baud_gen.v — Parameterized Baud Rate Generator
// Day 11, Exercise 1
// =============================================================================
// Produces a single-cycle tick at the baud rate.
// At 25 MHz / 115200 baud = 217 clocks per bit.

module baud_gen #(
    parameter CLK_FREQ  = 25_000_000,
    parameter BAUD_RATE = 115_200
)(
    input  wire i_clk,
    input  wire i_reset,
    input  wire i_enable,
    output wire o_tick
);

    localparam CLKS_PER_BIT = CLK_FREQ / BAUD_RATE;
    localparam CNT_WIDTH    = $clog2(CLKS_PER_BIT);

    // TODO: Implement a counter that:
    //   - Resets to 0 when i_reset or !i_enable
    //   - Counts up to CLKS_PER_BIT - 1, then wraps to 0
    //   - o_tick is high for exactly 1 cycle at the wrap point

    // ---- YOUR CODE HERE ----

endmodule

In [ ]:
%%writefile tb_baud_gen.v
// =============================================================================
// tb_baud_gen.v — Testbench for Baud Rate Generator
// Day 11, Exercise 1
// =============================================================================

`timescale 1ns / 1ps

module tb_baud_gen;

    reg  clk, reset, enable;
    wire tick;

    // Use small values for fast simulation
    localparam CLK_FREQ  = 100;
    localparam BAUD_RATE = 10;
    // Expected: tick every 10 clock cycles

    baud_gen #(
        .CLK_FREQ(CLK_FREQ),
        .BAUD_RATE(BAUD_RATE)
    ) uut (
        .i_clk(clk), .i_reset(reset),
        .i_enable(enable), .o_tick(tick)
    );

    initial clk = 0;
    always #5 clk = ~clk;  // 100 MHz sim clock

    integer tick_count, cycle_count;
    integer test_count = 0, fail_count = 0;

    initial begin
        $dumpfile("baud_gen.vcd");
        $dumpvars(0, tb_baud_gen);

        reset = 1; enable = 0;
        repeat (3) @(posedge clk);
        reset = 0; enable = 1;

        // TODO: Count cycles between ticks
        //   Run for 50 clock cycles, count how many ticks occur
        //   Expected: 5 ticks in 50 cycles (one every 10 cycles)
        //   Verify tick is exactly 1 cycle wide

        // ---- YOUR TEST CODE HERE ----

        tick_count = 0;
        for (cycle_count = 0; cycle_count < 50; cycle_count = cycle_count + 1) begin
            @(posedge clk); #1;
            if (tick) tick_count = tick_count + 1;
        end

        test_count = test_count + 1;
        if (tick_count != 5) begin
            fail_count = fail_count + 1;
            $display("FAIL: Expected 5 ticks in 50 cycles, got %0d", tick_count);
        end else begin
            $display("PASS: Tick count correct (%0d ticks)", tick_count);
        end

        // Test disable
        enable = 0;
        tick_count = 0;
        for (cycle_count = 0; cycle_count < 20; cycle_count = cycle_count + 1) begin
            @(posedge clk); #1;
            if (tick) tick_count = tick_count + 1;
        end

        test_count = test_count + 1;
        if (tick_count != 0) begin
            fail_count = fail_count + 1;
            $display("FAIL: Ticks while disabled (%0d)", tick_count);
        end else begin
            $display("PASS: No ticks while disabled");
        end

        $display("\n========================================");
        $display("Baud Gen: %0d/%0d tests passed",
                 test_count - fail_count, test_count);
        $display("========================================");
        $finish;
    end

endmodule

In [ ]:
%%writefile Makefile
# Makefile — baud_gen
PROJECT  = baud_gen
TOP      = baud_gen
PCF      = ../go_board.pcf
SRCS     = $(wildcard *.v)
TB       = tb_baud_gen.v

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_baud_gen.v baud_gen.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile tb_uart_tx.v
// =============================================================================
// tb_uart_tx.v — Protocol-Aware UART TX Testbench
// Day 11, Exercise 2
// =============================================================================

`timescale 1ns / 1ps

module tb_uart_tx;

    reg        clk, reset, valid;
    reg  [7:0] data;
    wire       tx, busy;

    localparam CLK_FREQ  = 1_000;  // 1 kHz for fast simulation
    localparam BAUD_RATE = 100;    // 100 baud → 10 clocks per bit
    localparam CLKS_PER_BIT = CLK_FREQ / BAUD_RATE;
    localparam CLK_PERIOD   = 500; // 500ns for 1 kHz

    uart_tx #(.CLK_FREQ(CLK_FREQ), .BAUD_RATE(BAUD_RATE)) uut (
        .i_clk(clk), .i_reset(reset),
        .i_valid(valid), .i_data(data),
        .o_tx(tx), .o_busy(busy)
    );

    initial clk = 0;
    always #(CLK_PERIOD/2) clk = ~clk;

    integer test_count = 0, fail_count = 0;

    // Task: capture one UART frame from the tx line
    task capture_uart_byte;
        output [7:0] captured;
        integer bit_idx;
    begin
        @(negedge tx);                           // wait for start bit
        #(CLKS_PER_BIT * CLK_PERIOD / 2);       // move to center of start bit

        if (tx !== 1'b0)
            $display("WARNING: Start bit not low at center");

        for (bit_idx = 0; bit_idx < 8; bit_idx = bit_idx + 1) begin
            #(CLKS_PER_BIT * CLK_PERIOD);        // advance to center of data bit
            captured[bit_idx] = tx;
        end

        #(CLKS_PER_BIT * CLK_PERIOD);            // move to center of stop bit
        if (tx !== 1'b1)
            $display("WARNING: Stop bit not high");
    end
    endtask

    // Task: send a byte and verify it
    task send_and_check;
        input [7:0] tx_byte;
        input [63:0] label;   // 8-char label
        reg [7:0] captured;
    begin
        @(posedge clk);
        data  = tx_byte;
        valid = 1;
        @(posedge clk);
        valid = 0;

        capture_uart_byte(captured);

        test_count = test_count + 1;
        if (captured !== tx_byte) begin
            fail_count = fail_count + 1;
            $display("FAIL [%0s]: sent=%h captured=%h", label, tx_byte, captured);
        end else begin
            $display("PASS [%0s]: %h OK", label, tx_byte);
        end

        @(negedge busy);
        repeat (5) @(posedge clk);
    end
    endtask

    initial begin
        $dumpfile("uart_tx.vcd");
        $dumpvars(0, tb_uart_tx);

        reset = 1; valid = 0; data = 0;
        repeat (5) @(posedge clk);
        reset = 0;
        repeat (5) @(posedge clk);

        // Test suite
        send_and_check(8'h41, "Letter A");
        send_and_check(8'h00, "NULL    ");
        send_and_check(8'hFF, "All 1s  ");
        send_and_check(8'h55, "Alt 0101");
        send_and_check(8'hAA, "Alt 1010");

        // Spell "HELLO"
        send_and_check(8'h48, "H       ");
        send_and_check(8'h45, "E       ");
        send_and_check(8'h4C, "L       ");
        send_and_check(8'h4C, "L       ");
        send_and_check(8'h4F, "O       ");

        $display("\n========================================");
        $display("UART TX: %0d/%0d tests passed",
                 test_count - fail_count, test_count);
        $display("========================================");
        $finish;
    end

endmodule

In [ ]:
%%writefile uart_tx.v
// =============================================================================
// uart_tx.v — UART Transmitter (8N1)
// Day 11, Exercise 2
// =============================================================================
// FSM + PISO shift register + baud counter.
// Frame: IDLE(high) → START(low) → 8 data bits (LSB first) → STOP(high)

module uart_tx #(
    parameter CLK_FREQ  = 25_000_000,
    parameter BAUD_RATE = 115_200
)(
    input  wire       i_clk,
    input  wire       i_reset,
    input  wire       i_valid,      // pulse high for 1 cycle to start TX
    input  wire [7:0] i_data,       // byte to transmit
    output reg        o_tx,         // serial output line
    output wire       o_busy        // high during transmission
);

    localparam CLKS_PER_BIT = CLK_FREQ / BAUD_RATE;
    localparam BAUD_CNT_W   = $clog2(CLKS_PER_BIT);

    // State encoding
    localparam S_IDLE  = 2'd0;
    localparam S_START = 2'd1;
    localparam S_DATA  = 2'd2;
    localparam S_STOP  = 2'd3;

    reg [1:0]            r_state;
    reg [BAUD_CNT_W-1:0] r_baud_cnt;
    reg [7:0]            r_shift;
    reg [2:0]            r_bit_idx;

    wire w_baud_tick = (r_baud_cnt == CLKS_PER_BIT - 1);

    assign o_busy = (r_state != S_IDLE);

    // TODO: Implement the FSM
    always @(posedge i_clk) begin
        if (i_reset) begin
            r_state    <= S_IDLE;
            o_tx       <= 1'b1;     // idle = high
            r_baud_cnt <= 0;
            r_bit_idx  <= 0;
            r_shift    <= 8'h00;
        end else begin
            case (r_state)
                S_IDLE: begin
                    o_tx       <= 1'b1;
                    r_baud_cnt <= 0;
                    r_bit_idx  <= 0;
                    if (i_valid) begin
                        r_shift <= i_data;
                        r_state <= S_START;
                    end
                end

                S_START: begin
                    // TODO: Drive o_tx low (start bit)
                    //       Count baud cycles
                    //       On baud_tick, transition to S_DATA

                    // ---- YOUR CODE HERE ----
                end

                S_DATA: begin
                    // TODO: Drive o_tx with r_shift[0] (LSB first)
                    //       Count baud cycles
                    //       On baud_tick:
                    //         Shift r_shift right by 1
                    //         Increment r_bit_idx
                    //         If r_bit_idx == 7, go to S_STOP

                    // ---- YOUR CODE HERE ----
                end

                S_STOP: begin
                    // TODO: Drive o_tx high (stop bit)
                    //       Count baud cycles
                    //       On baud_tick, return to S_IDLE

                    // ---- YOUR CODE HERE ----
                end
            endcase
        end
    end

endmodule

In [ ]:
%%writefile Makefile
# Makefile — uart_tx
PROJECT  = uart_tx
TOP      = uart_tx
PCF      = ../go_board.pcf
SRCS     = $(wildcard *.v)
TB       = tb_uart_tx.v

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_uart_tx.v uart_tx.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

#### 📝 Exercise 3 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile top_uart_hello.v
// =============================================================================
// top_uart_hello.v — Send "HELLO" on Button Press
// Day 11, Exercise 3: Hardware Verification
// =============================================================================

module top_uart_hello (
    input  wire i_clk,
    input  wire i_switch1,
    output wire o_uart_tx,
    output wire o_led1, o_led2, o_led3, o_led4
);

    // Debounce the button
    wire w_btn_clean;
    debounce #(.CLKS_TO_STABLE(250_000)) db (
        .i_clk(i_clk), .i_bouncy(i_switch1), .o_clean(w_btn_clean)
    );

    wire w_btn_press;
    // TODO: Edge detect — one-cycle pulse on button press
    //   Hint: reg r_prev; w_btn_press = ~w_btn_clean & ~r_prev_inverted
    //   (remember: active-low button, so pressed = 0)

    // ---- YOUR CODE HERE ----

    // UART TX instance
    wire w_tx_busy;
    reg  [7:0] r_tx_data;
    reg        r_tx_valid;

    uart_tx #(.CLK_FREQ(25_000_000), .BAUD_RATE(115_200)) tx (
        .i_clk(i_clk), .i_reset(1'b0),
        .i_valid(r_tx_valid), .i_data(r_tx_data),
        .o_tx(o_uart_tx), .o_busy(w_tx_busy)
    );

    // Message ROM: "HELLO\r\n"
    reg [7:0] r_message [0:6];
    initial begin
        r_message[0] = 8'h48;  // H
        r_message[1] = 8'h45;  // E
        r_message[2] = 8'h4C;  // L
        r_message[3] = 8'h4C;  // L
        r_message[4] = 8'h4F;  // O
        r_message[5] = 8'h0D;  // \r
        r_message[6] = 8'h0A;  // \n
    end

    // TODO: Implement a sequencer FSM
    //   States: IDLE, SEND, WAIT
    //   IDLE: wait for button press, then start sending
    //   SEND: assert r_tx_valid with r_message[index]
    //   WAIT: wait for !w_tx_busy, advance index
    //         if index == 7, return to IDLE

    // ---- YOUR CODE HERE ----

    // LED indicators
    assign o_led1 = ~w_btn_press;
    assign o_led2 = ~w_tx_busy;
    assign o_led3 = 1'b1;  // off
    assign o_led4 = 1'b1;  // off

endmodule

In [ ]:
%%writefile Makefile
# Makefile — top_uart_hello
PROJECT  = top_uart_hello
TOP      = top_uart_hello
PCF      = ../go_board.pcf
SRCS     = top_uart_hello.v uart_tx.v debounce.v
TB       = tb_top_uart_hello.v

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: $(PROJECT).bin

$(PROJECT).json: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $@" $(SRCS)

$(PROJECT).asc: $(PROJECT).json $(PCF)
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $< --asc $@

$(PROJECT).bin: $(PROJECT).asc
	icepack $< $@

prog: $(PROJECT).bin
	iceprog $<

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean